<a href="https://colab.research.google.com/github/acerNZ/HAL/blob/master/DevOpsMapReq_Userstories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================
# ONE-CLICK: Requirement → Feature → User Story Traceability
# Works with CSV or XLSX | Any column order | Auto-installs libraries
# Run this ENTIRE CELL in Google Colab → Upload 3 files → Done!
# ================================================

# --- STEP 0: Install & Import Required Libraries ---
import sys
import subprocess
import pkg_resources

def install_if_missing(package):
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

required = ["pandas", "numpy", "openpyxl", "ipywidgets"]
for pkg in required:
    install_if_missing(pkg)

import pandas as pd
import numpy as np
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML
import os

# --- STEP 1: Upload Files ---
print("\n" + "="*60)
print("UPLOAD YOUR 3 FILES (CSV or XLSX):")
print("1. User Stories")
print("2. Features")
print("3. Requirements")
print("="*60)

uploaded = files.upload()
if len(uploaded) != 3:
    raise ValueError("Please upload exactly 3 files!")

us_file = list(uploaded.keys())[0]
feat_file = list(uploaded.keys())[1]
req_file = list(uploaded.keys())[2]

print(f"\nLoaded:")
print(f"   • User Stories: {us_file}")
print(f"   • Features: {feat_file}")
print(f"   • Requirements: {req_file}")

# --- STEP 2: Load Files ---
def read_file(filepath):
    if filepath.endswith('.csv'):
        return pd.read_csv(filepath)
    else:
        return pd.read_excel(filepath)

us_df = read_file(us_file)
feat_df = read_file(feat_file)
req_df = read_file(req_file)

# --- STEP 3: Interactive Column Mapping ---
print("\n" + "="*60)
print("MAP COLUMNS BELOW (Use dropdowns)")
print("="*60)

def create_dropdown(df, label):
    return widgets.Dropdown(
        options=['-- Select --'] + df.columns.tolist(),
        description=label,
        layout=widgets.Layout(width='500px'),
        style={'description_width': '180px'}
    )

# User Stories
us_id = create_dropdown(us_df, "User Story ID")
us_title = create_dropdown(us_df, "US Title")
us_ac = create_dropdown(us_df, "Acceptance Criteria")
us_parent = create_dropdown(us_df, "Parent ID")
us_tags = create_dropdown(us_df, "Tags (optional)")

# Features
feat_id = create_dropdown(feat_df, "Feature ID")
feat_title = create_dropdown(feat_df, "Feature Title")

# Requirements
req_id = create_dropdown(req_df, "Requirement ID")
req_title = create_dropdown(req_df, "Req Title")
req_text = create_dropdown(req_df, "Req Text (optional)")
req_parent = create_dropdown(req_df, "Parent ID")
req_tags = create_dropdown(req_df, "Tags (optional)")

# Display
display(HTML("<h3>1. Map User Stories</h3>"))
display(us_id, us_title, us_ac, us_parent, us_tags)

display(HTML("<h3>2. Map Features</h3>"))
display(feat_id, feat_title)

display(HTML("<h3>3. Map Requirements</h3>"))
display(req_id, req_title, req_text, req_parent, req_tags)

print("\nAfter selecting all dropdowns → RUN THE NEXT CELL")

In [ ]:
# --- STEP 4: Validate & Extract Mapped Columns ---
print("\n" + "="*60)
print("EXTRACTING DATA...")
print("="*60)

def get_val(widget, name):
    if widget.value == '-- Select --':
        raise ValueError(f"Please select a column for: {name}")
    return widget.value

# Extract User Stories
us = pd.DataFrame({
    'US_ID': us_df[get_val(us_id, "User Story ID")].astype(str),
    'US_Title': us_df[get_val(us_title, "US Title")],
    'Acceptance_Criteria': us_df[get_val(us_ac, "Acceptance Criteria")].fillna(''),
    'US_Parent_ID': us_df[get_val(us_parent, "Parent ID")].astype(str),
    'US_Tags': us_df[us_tags.value].fillna('') if us_tags.value != '-- Select --' else pd.Series(['']*len(us_df))
})

# Extract Features
feat = pd.DataFrame({
    'Feature_ID': feat_df[get_val(feat_id, "Feature ID")].astype(str),
    'Feature_Title': feat_df[get_val(feat_title, "Feature Title")]
})

# Extract Requirements
req = pd.DataFrame({
    'Req_ID': req_df[get_val(req_id, "Requirement ID")].astype(str),
    'Req_Title': req_df[get_val(req_title, "Req Title")],
    'Req_Text': req_df[req_text.value].fillna('') if req_text.value != '-- Select --' else pd.Series(['']*len(req_df)),
    'Req_Parent_ID': req_df[get_val(req_parent, "Parent ID")].astype(str),
    'Req_Tags': req_df[req_tags.value].fillna('') if req_tags.value != '-- Select --' else pd.Series(['']*len(req_df))
})

print(f"Loaded: {len(us)} User Stories | {len(feat)} Features | {len(req)} Requirements")

In [ ]:
# --- STEP 5: Trace Requirement → Feature (Any Depth) ---
print("\n" + "="*60)
print("TRACING REQUIREMENTS → FEATURE (recursive, any depth)...")
print("="*60)

# Build parent lookup: Child → Parent
parent_map = {}
for _, row in req.iterrows():
    parent_map[row['Req_ID']] = row['Req_Parent_ID'] if row['Req_Parent_ID'] not in ('', 'nan') else None

# Add Features as top-level
for _, row in feat.iterrows():
    parent_map[row['Feature_ID']] = None

# Recursive function: find root Feature
def find_feature(req_id, visited=None):
    if visited is None:
        visited = set()
    if req_id in visited:
        return None  # loop
    visited.add(req_id)
    if req_id in feat['Feature_ID'].values:
        return req_id
    parent = parent_map.get(req_id)
    if not parent:
        return None
    return find_feature(parent, visited)

# Apply to all Requirements
req['Feature_ID'] = req['Req_ID'].apply(find_feature)

# Merge Feature Title
req = req.merge(feat[['Feature_ID', 'Feature_Title']], on='Feature_ID', how='left')
req.rename(columns={'Feature_Title': 'Feature_Name'}, inplace=True)

# --- STEP 6: Link to User Stories via Feature ---
print("LINKING USER STORIES via shared Feature...")
us_under_feat = us[us['US_Parent_ID'].isin(feat['Feature_ID'])]

# Final result: one row per Req + US
result = req[req['Feature_ID'].notna()].copy()
result = result.merge(
    us_under_feat[['US_ID', 'US_Title', 'Acceptance_Criteria', 'US_Parent_ID']],
    left_on='Feature_ID',
    right_on='US_Parent_ID',
    how='left'
).drop(columns=['US_Parent_ID'])

# Reorder & clean
final_cols = [
    'Req_ID', 'Req_Title', 'Req_Text', 'Req_Tags',
    'Feature_ID', 'Feature_Name',
    'US_ID', 'US_Title', 'Acceptance_Criteria'
]
result = result[final_cols].sort_values(['Req_ID', 'US_ID']).reset_index(drop=True)

# Fill blanks
for col in ['Req_Text', 'Req_Tags', 'Acceptance_Criteria']:
    result[col] = result[col].fillna('')

print(f"\nSUCCESS! {len(result)} rows in final table")
display(result.head(10))

# --- STEP 7: DOWNLOAD ---
output_file = "Requirement_to_UserStory_Traceability.xlsx"
result.to_excel(output_file, index=False)
files.download(output_file)

print(f"\nDOWNLOADED: {output_file}")
print("Run this notebook anytime — just upload new files!")